In [ ]:
import os
import pickle

import numpy as np
import torch
from matplotlib import colormaps
from matplotlib import pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from scipy.fft import fft

from tsfm_lens.chronos.circuitlens import CircuitLensChronos


In [ ]:
def sinusoidal(
    k: float | list[float],
    t_1: float = 1,
    steps: int = 100,
    amp: float | list[float] = 1,
    phase_shift: float = 0,
) -> tuple[np.ndarray, np.ndarray]:
    """Return t, y for sum of sinusoids with frequency k and amplitude amp."""
    t = np.linspace(0, t_1, steps, endpoint=False)
    k_arr = np.atleast_1d(k)
    amp_arr = np.full_like(k_arr, amp, dtype=float) if np.isscalar(amp) and len(k_arr) > 1 else np.atleast_1d(amp)
    if len(k_arr) != len(amp_arr):
        raise ValueError("k and amp must have the same length when passed as arrays")
    y = sum(a * np.sin(2 * np.pi * k_ * t + phase_shift) for k_, a in zip(k_arr, amp_arr))
    return t, y

In [ ]:
t, y = sinusoidal(k=[5, 10, 20], amp=[1, 2, 1.5], steps=512, t_1=1)
y.shape

In [ ]:
# plot y as univariate time series
plt.figure(figsize=(12, 3))
plt.plot(t, y)
plt.show()

In [ ]:
pipeline = CircuitLensChronos.from_pretrained("amazon/chronos-t5-base")
num_layers = pipeline.model.model.config.num_decoder_layers
num_heads = pipeline.model.model.config.num_heads
# pipeline.add_head_ablation_hooks([(layer,head) for layer in range(num_layers//4, num_layers) for head in range(num_heads)]);
pipeline.add_mlp_ablation_hooks([]);

In [ ]:
ablated_positions = set()
for layer in pipeline.ca_head_ablation_positions.keys():
    for head in pipeline.ca_head_ablation_positions[layer].keys():
        ablated_positions.add((layer, head))

ablated_positions;

# ablated_layers = pipeline.ca_head_ablation_positions.keys()

In [ ]:
context = torch.tensor(y).unsqueeze(0)
prediction_length = 1

preds, outputs = pipeline.predict_with_full_outputs(
    context=context,
    prediction_length=prediction_length,
    num_samples=1,
    do_sample=False,
    use_cache=False,
    return_dict_in_generate=True,
    output_attentions=True,
)

# (preds, outputs)
# output: (rollouts)
# rollouts: {return_dict}
# cross_attentions: (prediction_length, num_layers)
# cross_attention: [b, num_heads, pred_length, context_length+1]
rollout_idx = 0
target_token = 0

attns = {}
for layer in range(num_layers):
    attns[layer] = outputs[rollout_idx].cross_attentions[0][layer][0, :, target_token, :-1]

In [ ]:
# read in pickle file
model_name = "chronos-t5-base"
config_key = "rf8_sl64"

results_dir = "/work/results/rrt_induction_scores"
with open(os.path.join(results_dir, model_name, config_key, "center_scores.pkl"), "rb") as f:
    center_scores = pickle.load(f)
    center_scores_mean, center_scores_std = center_scores["mean"], center_scores["std"]

with open(os.path.join(results_dir, model_name, config_key, "right_scores.pkl"), "rb") as f:
    right_scores = pickle.load(f)
    right_scores_mean, right_scores_std = right_scores["mean"], right_scores["std"]

In [ ]:
induction_threshold = 0.9

colormap_name = "cividis_r"
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 12))

norm = Normalize(vmin=0, vmax=induction_threshold)
sm = ScalarMappable(norm=norm, cmap=colormaps[colormap_name])
sm.set_array([])

# --- Plot 1: Time domain ---
ax1.plot(t, y, "k-", linewidth=2, label="Sinusoidal Data", zorder=10)
ax1.set(ylabel="Data Amplitude", xlabel="Time", title="Sinusoidal Data and Attention Scores")
ax1.set_ylabel("Data Amplitude", fontweight="bold")
ax1.set_xlabel("Time", fontweight="bold")
ax1.set_title("Sinusoidal Data and Attention Scores", fontweight="bold")
ax1.tick_params(axis="y", labelcolor="k")
ax1.grid(True, alpha=0.3)
ax1.legend(loc="upper left", fontsize="small")

ax1_attn = ax1.twinx()
ax1_attn.set_ylabel("Attention Score", color="gray", fontweight="bold")
ax1_attn.tick_params(axis="y", labelcolor="gray")


high_induction_ffts, high_induction_labels = [], []
for layer in range(num_layers):
    for head in range(num_heads):
        if layer in pipeline.ca_head_ablation_positions and head in pipeline.ca_head_ablation_positions[layer]:
            continue
        score = right_scores_mean[layer, head]
        color = colormaps[colormap_name](norm(score))
        attn_vals = attns[layer][head].detach().cpu().numpy()
        ax1_attn.plot(t, attn_vals, alpha=0.3, linewidth=1, color=color)
        attn_fft = np.abs(fft(attn_vals))
        if score > induction_threshold:
            high_induction_ffts.append(attn_fft)
            high_induction_labels.append(f"L{layer}H{head}")

# --- Plot 2: Frequency domain ---
y_fft = np.abs(fft(y))
freqs = np.fft.fftfreq(len(t), t[1] - t[0])
pos_idx = np.where(freqs > 0)[0]

ax2.plot(
    freqs[pos_idx],
    y_fft[pos_idx],
    "k-",
    linewidth=2,
    label="Sinusoidal Data FFT",
    zorder=200,
)
ax2.set(
    ylabel="FT Magnitude",
    xlabel="Frequency",
    title="Fourier Transform of Data and Attention Scores",
    xlim=(0, 50),
)
ax2.set_ylabel("FFT Magnitude", fontweight="bold")
ax2.set_xlabel("Frequency", fontweight="bold")
ax2.set_title("Fourier Transform of Data and Attention Scores", fontweight="bold")
ax2.tick_params(axis="y", labelcolor="k")
ax2.grid(True, alpha=0.3)
ax2.legend(loc="upper left", fontsize="small")

ax2_attn = ax2.twinx()
ax2_attn.tick_params(axis="y", labelcolor="gray")
for layer in range(num_layers):
    for head in range(num_heads):
        if layer in pipeline.ca_head_ablation_positions and head in pipeline.ca_head_ablation_positions[layer]:
            continue
        score = right_scores_mean[layer, head]
        color = colormaps[colormap_name](norm(score))
        attn_fft = np.abs(fft(attns[layer][head].detach().cpu().numpy()))
        ax2_attn.plot(
            freqs[pos_idx],
            attn_fft[pos_idx],
            alpha=0.2,
            linewidth=1,
            zorder=1,
            color=color,
        )

# --- Plot 3: FFT of FFT for high induction score heads ---
ax3.set(
    title="FFT of FFT of High Induction Score Heads (>0.3)",
    xlabel="Frequency",
    ylabel="FFT of FFT Magnitude",
    # ylim=(0, 30),
)
ax3.set_title(
    f"FFT of FFT of High Induction Score Heads (> {induction_threshold})",
    fontweight="bold",
)
ax3.set_xlabel("Frequency", fontweight="bold")
ax3.set_ylabel("FFT of FFT Magnitude", fontweight="bold")
ax3.grid(True, alpha=0.3)

for fft_data, label in zip(high_induction_ffts, high_induction_labels):
    # Take the FFT of the FFT magnitude
    fft_of_fft = np.abs(np.fft.fft(fft_data))
    # Use the same frequency axis as the original data
    ax3.plot(freqs[pos_idx], fft_of_fft[pos_idx], label=label, linewidth=1.5)

if high_induction_ffts:
    ax3.legend(loc="upper right", fontsize="small", title="Attention Heads")
else:
    ax3.text(
        0.5,
        0.5,
        "No heads with induction score > 0.3",
        ha="center",
        va="center",
        transform=ax3.transAxes,
        fontsize=12,
    )

fig.colorbar(sm, cax=fig.add_axes([0.92, 0.15, 0.02, 0.7])).set_label("Induction Score", fontweight="bold")
plt.tight_layout(rect=(0, 0, 0.9, 1))
plt.show()


In [ ]:
# freqs[positive_freq_idxs]
# second_order_fft[positive_freq_idxs].shape